# narwhals-datafusion playground

Narwhals expressions on a DataFusion frame. Everything is lazy until `collect()`.

In [ ]:
%pip install --upgrade narwhals-datafusion

In [ ]:
import pyarrow as pa
from datafusion import SessionContext

import narwhals as nw

ctx = SessionContext()
sales = ctx.from_arrow(
    pa.table(
        {
            "region": ["north", "north", "south", "south", "south", "west"],
            "day": [1, 2, 1, 2, 3, 1],
            "units": [10, 12, 7, None, 9, 4],
            "price": [2.5, 2.5, 3.0, 3.0, 3.25, 5.0],
        }
    )
)
lf = nw.from_native(sales)
lf

The schema comes from the plan, not from data. Dtypes are pyarrow end to end.

In [ ]:
lf.collect_schema()

## Expressions

`select`, `filter` and `with_columns` add projection and filter nodes to the plan.

In [ ]:
revenue = (
    lf.with_columns(revenue=nw.col("units") * nw.col("price"))
    .filter(~nw.col("units").is_null())
    .select("region", "day", "revenue")
    .sort("region", "day")
)
revenue.collect()

## Aggregates and windows

A bare aggregate reduces the frame; the same aggregate with `over` becomes a window.

In [ ]:
lf.group_by("region").agg(
    units=nw.col("units").sum(),
    days=nw.len(),
    avg_price=nw.col("price").mean(),
).sort("region").collect()

In [ ]:
lf.with_columns(
    region_units=nw.col("units").sum().over("region"),
    running=nw.col("units").cum_sum().over("region", order_by="day"),
    filled=nw.col("units").fill_null(strategy="forward").over(order_by=["region", "day"]),
).sort("region", "day").collect()

## Joins and concat

Column names that collide get the `_right` suffix. Any name works, including mixed case and dots.

In [ ]:
managers = nw.from_native(
    ctx.from_arrow(
        pa.table({"region": ["north", "south"], "price": [1.0, 2.0], "Manager.Name": ["ann", "bo"]})
    )
)
lf.join(managers, on="region", how="left").sort("region", "day").collect()

In [ ]:
extra_rows = nw.from_native(ctx.from_arrow(pa.table({"region": ["east"], "units": [3]})))
nw.concat([lf, extra_rows], how="diagonal").sort("region", "day", nulls_last=True).collect()

## Reshaping

In [ ]:
lf.unpivot(on=["units", "price"], index=["region", "day"]).sort("region", "day", "variable").head(
    6
).collect()

In [ ]:
lists = nw.from_native(ctx.from_arrow(pa.table({"id": [1, 2, 3], "tags": [["a", "b"], [], None]})))
lists.explode("tags").sort("id", "tags", nulls_last=True).collect()

## The plan behind it

`to_native()` returns the `datafusion.DataFrame`; nothing has executed yet.

In [ ]:
print(revenue.to_native().logical_plan().display_indent())

## Collecting elsewhere

The result lands in pyarrow by default; pandas and polars are one keyword away.

In [ ]:
revenue.collect(backend="polars").to_native()

## The `extra-functions` extra

`mode`, `skew` and `kurtosis` come from a Rust crate loaded through DataFusion's FFI.

In [ ]:
%pip install --upgrade "narwhals-datafusion[extra-functions]"

In [ ]:
lf.select(
    mode=nw.col("price").mode(keep="any"),
    skew=nw.col("price").skew(),
    kurtosis=nw.col("price").kurtosis(),
).collect()

## What is refused

Where DataFusion would return a plausible wrong answer, the backend raises instead. `DISTINCT` inside a window aggregate is dropped by the engine, so `n_unique` over a window is not offered.

In [ ]:
try:
    lf.select(nw.col("units").n_unique().over("region"))
except NotImplementedError as error:
    print(error)